# Benchmarking to MODFLOW

This notebook provides a comparison of concentrations calculated with the *mibitrans* package and concentrations calculated with the numerical model *MODFLOW 2005*$^{1}$. The notebook benchmarks *mibitrans* showing that equation implementations are correct.

**Authors:** Jorrit Bakker, Alraune Zech


## Background

We make use of the *flopy* Python-API for MODFLOW*$^{1}$ that solve the groundwater flow equation numerically and the transport equation MT3DMS package of the MODFLOW family. Calculations are based on the finite difference method. For the same choice of initial and boundary conditions, both the model class `Mibitrans` of the *mibitrans* package and MODFLOW should provide the same results.

**References**

$^1$ Harbaugh, A. W., MODFLOW-2005, the U.S. geological survey modular ground-water model – the ground-water flow process, Tech. Rep. 6-A16, U.S. Geological Survey Techniques and Methods, 2005.

## NOTE TO USER:

Running this notebook requires the additional dependency of having the executables for running MODFLOW-2005 and MT3DMS installed. See the [flopy-documentation](https://flopy.readthedocs.io/en/latest/md/get_modflow.html) for more information. Paths to the executables have been set to default values in the cell below, but should be manually set to the executable location on your machine if required.

In [ ]:
# Set the paths to executable locations for MODFLOW and MT3DMS
EXE_NAME_MF = 'mf2005'
EXE_NAME_MS = 'mt3dms'

In [ ]:
import flopy as fp
import matplotlib.pyplot as plt
import numpy as np
import mibitrans as mbt
from mibitrans.analysis.to_modflow import MibitransToModflow

## Benchmarking

### Example dataset

Example values used in this comparison are those used for the Keesler model (see `example_keesler.ipynb` for more information on the example parameters), except for a zero decay rate.

Note that for the benchmarking, we will be using a fine model resolution, so model might take a few minutes to run.

In [ ]:
hydro = mbt.HydrologicalParameters(
    h_conductivity = 9.504, # [m/d]
    h_gradient = 0.003, # [m/m]
    porosity = 0.3, # [-]
    alpha_x = 10, # [m] (Note; high dispersivity values as using same input Keesler example)
    alpha_y = 1, # [m]
    alpha_z = 0 # [m]
)
att = mbt.AttenuationParameters(
    bulk_density=1700,             # [kg/m^3]
    partition_coefficient=0.038,   # [m^3/kg] (units cancel out with bulk density)
    fraction_organic_carbon=0.000057, # [-]
    decay_rate = 0,                # [1/day]
)
source = mbt.SourceParameters(# Note; as plume is symmetric, source is described from center outwards in y-direction.
    source_zone_boundary=np.array([2, 11, 20]), # [m]
    source_zone_concentration=np.array([13.68, 2.508, 0.057]), #[g/m3]
    depth=3, # [m]
    total_mass=np.inf, # [g]
)
model = mbt.ModelParameters(
    model_length=300, # [m]
    model_width=60, # [m]
    model_time=6*365, # [days]
    dx=0.25, # [m]
    dy=0.25, # [m]
    dt=365/5 # [days]
)

### Running mibitrans-model

In [ ]:
mbt_model = mbt.Mibitrans(hydro, att, source, model)
mbt_results = mbt_model.run()
mbt_results.centerline()

### Converting *mibitrans*-input to MODLFOW-input

The `to_modflow` module of *mibitrans* allows to convert *mibitrans* dataclass input parameters into input for a combined MODFLOW-2005 and MT3DMS model.

Input for conversion to MODFLOW is the same as input for the analytical models. But as mibitrans gives output at grid-coordiantes as opposed MODFLOW (and numerical models in general) at the cell centers, we gave a 2x finer grid to mibitrans, so we index at the exact same locations at the comparison later (if not for the quantitative comparison, there is no need to perform any adaptions to the resolution.

The `MibitransToModflow()` class gives an object with flopy input as attributes that can be adapted (e.g. use `object.top` to change/check the depth of the model). Using `object.to_modflow()` creates flopy `mf` and `mt` objects, containing the input parameters. Consequently, any adaptations or additions to model can be performed on these objects. In the example below, the Btn package is overwritten to include additional output times for concentrations.

In [ ]:
mf_mb_dx_ratio = 2
mf_mb_dy_ratio = 2

model_mf = mbt.ModelParameters(
    model_length=model.model_length, # [m]
    model_width=model.model_width, # [m]
    model_time=model.model_time, # [days]
    dx=model.dx*mf_mb_dx_ratio, # [m]
    dy=model.dy*mf_mb_dy_ratio, # [m]
    dt=model.dt # [days]
)

In [ ]:
mbt_mf = MibitransToModflow(hydro, att, source, model_mf)
mbt_mf.to_modflow()

# Set the executables for MODFLOW and MT3DMS to user provided paths (from first cell in this notebook)
mbt_mf.exe_name_mf = EXE_NAME_MF
mbt_mf.exe_name_ms = EXE_NAME_MS
mf = mbt_mf.mf
mt = mbt_mf.mt
# Note: the line below changes the mt object in place, thus, it adapts the mt object inside the mbt_mf object.
mt3d_btn = fp.mt3d.Mt3dBtn(
    mt,
    icbund=mbt_mf.icbund,
    prsity=mbt_mf.prsity,
    sconc=mbt_mf.sconc,
    nprs=len(mbt_results.t),
    timprs=mbt_results.t
)

### Running MODFLOW Model

Use the `run_modflow()` method to run the model, which returns the concentration distribution and output times. (this step requires the MODFLOW-2005 and MT3DMS executables). Instead, input can be written to MODFLOW input files using `mf.write_input()` and/or `mt.write_input()` and be used elsewhere.

In [ ]:
conc, times = mbt_mf.run_modflow()

Use the `centerline_modflow()` method to quickly visualize the calculated concentration distribution over the plume centerline for the last timestep.

In [ ]:
mbt_mf.centerline_modflow()
plt.show()

### Model comparison

The streamlined model setup allows simple comparison of both, the analytical and the numerical model results (for the same set of input parameters and flow/transport settings).

In [ ]:
# Determining location plume for plotting
x_pos = 80 # x-position in [m] for transverse plot
plot_times = [1,2,4,6] # at which years to plot

x_pos_mbt = int(x_pos*(1/model.dx))+mf_mb_dx_ratio//2 # x-position index for transverse plot
x_pos_mf = int(x_pos*(1/model_mf.dx))

x_mbt = mbt_results.x
y_mbt = mbt_results.y
dt_to_yr = 365 // model.dt
model_middle = len(y_mbt)//2
row = mf.modelgrid.nrow // 2
x_mf = mf.modelgrid.xcellcenters[row, :]
y_mf = mf.modelgrid.ycellcenters[:, 0]

# Plot colors
cmap = plt.get_cmap('tab20b')
colors = cmap.colors
ci = 4
co = 2
fig, ax = plt.subplots(figsize=(10, 4), ncols=2)
for i, t in enumerate(plot_times):
    yr_id = int(t*dt_to_yr-1)
    ax[0].plot(x_mbt, mbt_model.cxyt[yr_id, model_middle, :], color=colors[ci*i+co], lw=3, label=f"t={t}y", alpha=0.3)
    ax[1].plot(y_mbt, mbt_model.cxyt[yr_id, :, x_pos_mbt], color=colors[ci*i+co], lw=3, alpha=0.3)
for i, t in enumerate(plot_times):
    yr_id = int(t*dt_to_yr-1)
    ax[0].plot(x_mf-0.25, conc[yr_id, 0, row, :], linestyle="--", color=colors[ci*i+co], lw=3, label=" ")
    ax[1].plot(y_mf-60, conc[yr_id, 0, :, x_pos_mf], linestyle="--", color=colors[ci*i+co], lw=3,)
ax[1].set_xlim((-40,40))
ax[0].legend(title="   mibitrans    MODFLOW", ncol=2)
ax[0].set_xlabel("Distance from source [m]")
ax[1].set_xlabel("Distance from plume center [m]")
ax[0].set_ylabel(r"Concentration [$g/m^3$]")
plt.show()

Visual comparison of output from *mibitrans* and MODFLOW indicates nearly identical results - as is expected.

The only noteable difference is near the model boundaries. Model boundaries are only relevant for the analytical models regarding the visualization domain. For the numerical model constant head boundary conditions introduce a slight deviation. Picking a model extent larger than the relevant plume area would prevent these minor differences.

### Quantitative comparison

In [ ]:
### Functions for quantitative comparison
from mibitrans.analysis import differences as diff

To make a quantitative comparison, we have to ensure that discretization of the concentration arrays match. Therefore we construct a discretization that contains the same points for both models. Then, check if they indeed correspond to eachother.

In [ ]:
y_domain = len(y_mf)//4
mf_y_domain = model.model_width - y_mf[y_domain:-y_domain]
mb_y_domain = mbt_results.y[mf_mb_dy_ratio//2::mf_mb_dy_ratio]
middle = int((model_mf.model_width // model_mf.dy) // 2)
print("Adapted MODFLOW y-discretization", mf_y_domain[middle - 3:middle + 3])
print("Adapted mibitrans y-discretization", mb_y_domain[middle - 3:middle + 3:])
print("Total difference y-discretization", np.sum(np.abs(mf_y_domain - mb_y_domain)))
mf_x_domain = x_mf
mb_x_domain = mbt_results.x[mf_mb_dx_ratio//2::mf_mb_dx_ratio]
print("MODFLOW x-discretization", mf_x_domain[:5])
print("Adapted mibitrans x-discretization", mb_x_domain[:5])
print("Total difference x-discretization", np.abs(np.sum(mf_x_domain - mb_x_domain)))

Active flow and transport domain in MODFLOW only begins at x=dx, while for mibitrans, it only starts at x=0. Therefore, we include x=0 in the domain array. Then, source locations and sampling points of both models should align with proper indexing.

In [ ]:
comp_x_domain = np.concatenate(([mbt_results.x[0]], mb_x_domain))

Index and slice the concentration arrays of both models the same ways as their x and y domains.

In [ ]:
mf_comp_cxyt = conc[:,0,y_domain:-y_domain,:]
mb_comp_cxyt = mbt_results.cxyt[:,mf_mb_dy_ratio//2::mf_mb_dy_ratio,mf_mb_dx_ratio//2:-mf_mb_dx_ratio:mf_mb_dx_ratio]
mb_source = mbt_results.cxyt[:,mf_mb_dy_ratio//2::mf_mb_dy_ratio,0]
mb_comp_cxyt = np.concatenate((mb_source[:,:,None], mb_comp_cxyt), axis=2)
mb_comp_cxyt_rel = mbt_results.relative_cxyt[
    :,
    mf_mb_dy_ratio//2::mf_mb_dy_ratio,
    mf_mb_dx_ratio//2:-mf_mb_dx_ratio:mf_mb_dx_ratio
]
mb_source_rel = mbt_results.relative_cxyt[:,mf_mb_dy_ratio//2::mf_mb_dy_ratio,0]
mb_comp_cxyt_rel = np.concatenate((mb_source_rel[:,:,None], mb_comp_cxyt_rel), axis=2)

Confirm that source zones match.

In [ ]:
print("Mean difference in source zone concentrations between MODFLOW and mibitrans is:",
      np.mean(np.abs(mf_comp_cxyt[-1,:,0] - mb_comp_cxyt[-1,:,0])), "g/m3")

Then, calculate the mean difference of time and space between the two models. To account for boundary effect, we do not consider the last 20 cells.

In [ ]:
mad_mibi_mod = diff.mean_absolute_difference(
    mf_comp_cxyt[:,:,:-20] / np.max(source.source_zone_concentration),
    mb_comp_cxyt_rel[:,:,:-20], mean_axis=(0,1,2)
)
print(mad_mibi_mod*100)

### Visualization of quantitative differences

Below, differences between MODFLOW and mibitrans are visualized in various ways.

In [ ]:
fontsize=14

In [ ]:
mad_mibi_mod_xy = diff.mean_absolute_difference(
    mf_comp_cxyt / np.max(source.source_zone_concentration),
    mb_comp_cxyt_rel, mean_axis=(0)
)
plt.figure(figsize=(12,6))
plt.pcolormesh(comp_x_domain[:-1], mb_y_domain, np.log10(np.abs(mad_mibi_mod_xy)), cmap="viridis")
plt.xlabel("Distance from source [m]", fontsize=fontsize)
plt.ylabel("Distance from plume center [m]", fontsize=fontsize)
plt.colorbar(label=r"$log_{10}$ of mean absolute difference in concentration [$g/m^3$]")
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
plt.pcolormesh(comp_x_domain[:-1], mb_y_domain, mad_mibi_mod_xy, vmax=0.01, cmap="viridis")
plt.xlabel("Distance from source [m]", fontsize=fontsize)
plt.ylabel("Distance from plume center [m]", fontsize=fontsize)
plt.colorbar(label=r"Mean difference in concentration [$g/m^3$]")
plt.show()

In [ ]:
time_index = -1
c_diff = mb_comp_cxyt[time_index,:,:] - mf_comp_cxyt[time_index,:,:]

plt.figure(figsize=(12,6))
plt.pcolormesh(comp_x_domain[:-1], mb_y_domain, c_diff, vmin=-0.1, vmax=0.1, cmap="RdBu_r")
plt.xlabel("Distance from source [m]", fontsize=fontsize)
plt.ylabel("Distance from plume center [m]", fontsize=fontsize)
plt.colorbar(label=r"Difference in concentration [$g/m^3$] at t=6y")
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
plt.pcolormesh(comp_x_domain[:-1], mb_y_domain, np.log10(np.abs(c_diff)), cmap="YlOrRd")
plt.xlabel("Distance from source [m]", fontsize=fontsize)
plt.ylabel("Distance from plume center [m]", fontsize=fontsize)
plt.colorbar(label=r"Absolute difference in concentration [$g/m^3$] at t=6y")
plt.show()

Differences between the two models is sensitive to the dx and dy, which is expected for a numerical model.

Thus, `mibitrans` gives very similar output as a simple numerical model, and offers an easy way to generate a setup of a numerical model under the same conditions.